In [1]:
import random
import math
import numpy as np
import torch

SEED = 42 #Setting the global seed to 42 to guarantee reproducible results

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available(): #Forcing deterministic behavior on the GPU to avoid floating-point discrepancies
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False #Disabling the auto-tuner to guarantee exact mathematical reproducibility

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute Device Active: {device}")

Compute Device Active: cuda


In [2]:
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

X_train_np = np.load('Data/X_train_processed.npy')
y_train_np = np.load('Data/y_train_processed.npy')
engine_ids = np.load('Data/engine_ids_train.npy')
X_test_np = np.load('Data/X_test_processed.npy')
y_test_np = np.load('Data/y_test_processed.npy')

X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32) #Converting numpy arrays into 32-bit tensors to move the math to the GPU for faster training
y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_np, dtype=torch.float32)

unique_engines = np.unique(engine_ids) #Shuffling engines randomly but reproducibly before splitting (80/20)
np.random.seed(42)
np.random.shuffle(unique_engines)

train_engine_count = int(0.8 * len(unique_engines)) #Assigning 80% of engines to training, 20% to the clean validation quarantine
train_engines = set(unique_engines[:train_engine_count])
val_engines = set(unique_engines[train_engine_count:])

train_mask = np.array([eid in train_engines for eid in engine_ids]) #Building boolean masks to guarantee sequences from the same engine never cross the boundary
val_mask   = ~train_mask

train_dataset = TensorDataset(X_train_tensor[train_mask], y_train_tensor[train_mask]) #Zipping the inputs and targets together and shuffling them in batches of 64 size so the model learns patterns instead of memorizing the order
val_dataset   = TensorDataset(X_train_tensor[val_mask],   y_train_tensor[val_mask])
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"X_train Tensor Shape: {X_train_tensor.shape}")
print(f"y_train Tensor Shape: {y_train_tensor.shape}")
print(f"Engines in training: {len(train_engines)} , Engines in validation: {len(val_engines)}")

X_train Tensor Shape: torch.Size([53779, 30, 17])
y_train Tensor Shape: torch.Size([53779])
Engines in training: 199 , Engines in validation: 50


In [3]:
import torch.nn as nn

class AttentionBiLSTM(nn.Module):
    def __init__(self, input_size=17, hidden_size=128, num_layers=2, dropout_rate=0.3):
        super(AttentionBiLSTM, self).__init__()

        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True, dropout=dropout_rate) #Increased hidden_size to 128 to give the AI more brain capacity for the FD004 dataset
        
        self.attention_scoring_layer = nn.Linear(hidden_size * 2, 1) #A linear layer to grade the importance of each cycle's mathematical thoughts
        
        self.intermediate_decision_layer = nn.Linear(hidden_size * 2, 64) #Translating 256 LSTM outputs into 64 features before prediction
        self.relu_activation = nn.ReLU() #Allows the network to learn non-linear degradation patterns
        self.dropout_layer = nn.Dropout(dropout_rate) #Heavy dropout to prevent memorization during the long 60-epoch run
        self.final_decision_layer = nn.Linear(64, 1)

    def forward(self, x):
        timeline_memory, _ = self.lstm(x)
        raw_scores = self.attention_scoring_layer(timeline_memory) #Calculating the raw importance scores for all 30 cycles
        attention_weights = torch.softmax(raw_scores, dim=1) #Normalizing the scores into percentages that sum to 1.0
        context_vector = torch.sum(attention_weights * timeline_memory, dim=1) #Multiplying the memory by the percentages to create the context vector
        
        out = self.relu_activation(self.intermediate_decision_layer(context_vector))
        out = self.dropout_layer(out)
        rul_prediction = self.final_decision_layer(out)
        
        return rul_prediction, attention_weights

In [4]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

final_epochs = 60
attention_model = AttentionBiLSTM(hidden_size=128, dropout_rate=0.25).to(device)
adam_optimizer = optim.Adam(attention_model.parameters(), lr=0.001, weight_decay=1e-4) #L2 Regularization (Weight Decay) added to prevent the model from memorizing the training data
huber_calculator = nn.HuberLoss(delta=2.0) #Huber Loss acts as a shock absorber against chaotic sensor outliers near failure
mse_calculator = nn.MSELoss()

scheduler = ReduceLROnPlateau(adam_optimizer, mode='min', patience=5, factor=0.5) #The Automatic Transmission: If validation loss plateaus for 5 consecutive epochs, the learning rate is halved — forcing finer steps as the model approaches the error floor

for epoch in range(final_epochs):
    attention_model.train()
    total_train_error = 0.0
    
    for batch_sensor_data, actual_rul in train_loader:
        batch_sensor_data = batch_sensor_data.to(device)
        actual_rul = actual_rul.to(device)
        
        adam_optimizer.zero_grad()
        
        predicted_rul, _ = attention_model(batch_sensor_data)
        predicted_rul = predicted_rul.squeeze(-1)
        
        batch_error = huber_calculator(predicted_rul, actual_rul)
        batch_error.backward()
        
        torch.nn.utils.clip_grad_norm_(attention_model.parameters(), max_norm=1.0) #Forcing a hard speed limit on gradients to prevent LSTM destabilization
        
        adam_optimizer.step()
        
        total_train_error += batch_error.item()
        
    average_train_error = total_train_error / len(train_loader)

    attention_model.eval() #Grading the model on unseen validation engines every single epoch so the scheduler knows if the model is actually generalizing
    val_error = 0.0
    
    with torch.no_grad():
        for val_sensor_data, val_actual_rul in val_loader:
            val_sensor_data = val_sensor_data.to(device)
            val_actual_rul = val_actual_rul.to(device)
            
            val_predictions, _ = attention_model(val_sensor_data)
            val_predictions = val_predictions.squeeze(-1)
            val_loss = mse_calculator(val_predictions, val_actual_rul)
            val_error += val_loss.item()
            
    average_val_error = val_error / len(val_loader)

    attention_model.train()
    scheduler.step(average_val_error) #The scheduler legitimately adjusts the learning rate based strictly on unseen validation engines
    
    current_lr = adam_optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch+1:02d}/{final_epochs} , Train Huber: {average_train_error:.2f} , Val MSE: {average_val_error:.2f} , LR: {current_lr:.6f}")


attention_model.eval()
total_sq_error = 0.0
total_samples = 0

with torch.no_grad():
    for batch_sensor_data, actual_rul in test_loader:
        batch_sensor_data = batch_sensor_data.to(device)
        actual_rul = actual_rul.to(device)
        
        predicted_rul, _ = attention_model(batch_sensor_data)
        predicted_rul = predicted_rul.squeeze(-1)
        
        sq_error = ((predicted_rul - actual_rul) ** 2).sum().item()
        total_sq_error += sq_error
        total_samples += actual_rul.size(0)

final_test_mse = total_sq_error / total_samples
final_rmse = math.sqrt(final_test_mse)

print(f"\nATTENTION OPTIMIZED RESULTS")
print(f"Final Test MSE: {final_test_mse:.2f}")
print(f"Final Test RMSE: {final_rmse:.2f} cycles")

torch.save(attention_model.state_dict(), 'Data/attention_turbofan_model.pth')

Epoch 01/60 , Train Huber: 58.53 , Val MSE: 304.89 , LR: 0.001000
Epoch 02/60 , Train Huber: 29.81 , Val MSE: 248.43 , LR: 0.001000
Epoch 03/60 , Train Huber: 28.12 , Val MSE: 232.17 , LR: 0.001000
Epoch 04/60 , Train Huber: 27.91 , Val MSE: 209.32 , LR: 0.001000
Epoch 05/60 , Train Huber: 27.00 , Val MSE: 246.76 , LR: 0.001000
Epoch 06/60 , Train Huber: 26.20 , Val MSE: 207.92 , LR: 0.001000
Epoch 07/60 , Train Huber: 25.60 , Val MSE: 228.97 , LR: 0.001000
Epoch 08/60 , Train Huber: 25.41 , Val MSE: 195.26 , LR: 0.001000
Epoch 09/60 , Train Huber: 24.73 , Val MSE: 194.86 , LR: 0.001000
Epoch 10/60 , Train Huber: 24.42 , Val MSE: 194.83 , LR: 0.001000
Epoch 11/60 , Train Huber: 23.96 , Val MSE: 183.88 , LR: 0.001000
Epoch 12/60 , Train Huber: 23.74 , Val MSE: 176.02 , LR: 0.001000
Epoch 13/60 , Train Huber: 23.60 , Val MSE: 176.84 , LR: 0.001000
Epoch 14/60 , Train Huber: 23.41 , Val MSE: 190.09 , LR: 0.001000
Epoch 15/60 , Train Huber: 23.10 , Val MSE: 185.33 , LR: 0.001000
Epoch 16/6

In [5]:
attention_model.eval()

extracted_weights_list = []
extracted_predictions_list = []

with torch.no_grad():
    for batch_sensor_data, _ in test_loader:
        batch_sensor_data = batch_sensor_data.to(device)
        
        predicted_rul, batch_attention_weights = attention_model(batch_sensor_data) #Collecting both RUL predictions and cycle-level attention weights for each test engine in one forward pass    
        extracted_weights_list.append(batch_attention_weights.cpu().numpy().squeeze(-1)) #Transferring data from GPU to RAM, converting to NumPy, and removing empty dimensions
        extracted_predictions_list.append(predicted_rul.cpu().numpy().squeeze(-1))


final_attention_weights = np.concatenate(extracted_weights_list, axis=0) #Stacking all batches into one continuous array
final_predictions = np.concatenate(extracted_predictions_list, axis=0)

print(f"Shape of Attention Weights: {final_attention_weights.shape}")
print(f"Shape of Predicted RULs: {final_predictions.shape}")

np.save('Data/attention_weights.npy', final_attention_weights)
np.save('Data/predicted_rul_attention.npy', final_predictions)

Shape of Attention Weights: (248, 30)
Shape of Predicted RULs: (248,)


In [6]:
import shap

baseline_sample_count = 200 #Setting 200 samples for our normal engine baseline to balance statistical stability and GPU VRAM limits

np.random.seed(42)

train_indices = np.where(train_mask)[0]
background_engine_indices = np.random.choice(train_indices, baseline_sample_count, replace=False) #Sampling 200 sequence indices exclusively from the training partition so that the validation data never enters the SHAP baseline
background_reference_tensor = X_train_tensor[background_engine_indices].to(device) #Extracting baseline 3D sensor arrays and transferring to GPU

print(f"Shape: {background_reference_tensor.shape}")

Shape: torch.Size([200, 30, 17])


In [7]:
class SHAPModelAdapter(torch.nn.Module): #Adapter Class to intercept the BiLSTM tuple and return only the RUL prediction for SHAP compatibility
    def __init__(self, core_model):
        super(SHAPModelAdapter, self).__init__()
        self.core_model = core_model

    def forward(self, x):
        rul_prediction, _ = self.core_model(x)
        return rul_prediction


shap_compatible_model = SHAPModelAdapter(attention_model).to(device) 

shap_compatible_model.eval() #Forcing eval mode to prevent Dropout interference

shap_diagnostic_tool = shap.GradientExplainer(shap_compatible_model, background_reference_tensor) #Using GradientExplainer because DeepExplainer has documented architectural conflicts when backpropagating through cuDNN's optimized BiLSTM kernels
test_sensor_data_gpu = X_test_tensor.to(device) #Loading unseen test engines to GPU for final evaluation

torch.backends.cudnn.enabled = False #Temporarily disabling NVIDIA's cuDNN engine to bypass the RNN eval-mode gradient restriction
shap_values = shap_diagnostic_tool.shap_values(test_sensor_data_gpu) #Executing SHAP calculations to extract the 3D array of RUL impacts
torch.backends.cudnn.enabled = True #Re-enabling cuDNN to return the GPU to its normal state

print("Matrix calculations complete!")

Matrix calculations complete!


In [8]:
final_shap_array = np.array(shap_values)

if len(final_shap_array.shape) == 4 and final_shap_array.shape[-1] == 1: #Specifically targeting the final axis to destroy the useless classification wrapper without risking dimensional collapse on single-engine test batches
    final_shap_array = np.squeeze(final_shap_array, axis=-1)

print(f"Final SHAP Array Geometry: {final_shap_array.shape}")

np.save('Data/shap_values.npy', final_shap_array)

sensor_importance = np.mean(np.abs(final_shap_array), axis=(0, 1)) #Collapsing the Engine and Time dimensions to extract the absolute overall impact of each sensor
np.save('Data/shap_sensor_importance.npy', sensor_importance)

early_life_mask = y_test_np >= 80  #Isolating specific thermodynamic degradation stages using true Remaining Useful Life labels
late_life_mask  = y_test_np <= 30

shap_early_life = np.mean(np.abs(final_shap_array[early_life_mask]), axis=(0, 1)) #Extracting the targeted 1D sensor arrays for healthy vs. near-failure states
shap_late_life  = np.mean(np.abs(final_shap_array[late_life_mask]),  axis=(0, 1))

np.save('Data/shap_early_life.npy', shap_early_life)
np.save('Data/shap_late_life.npy',  shap_late_life)

print(f"Early-life engines: {early_life_mask.sum()} , Late-life engines: {late_life_mask.sum()}")

Final SHAP Array Geometry: (248, 30, 17)
Early-life engines: 135 , Late-life engines: 53
